# Locality-iN-Locality-Ti: GTSRB training, validation and test top-1

This notebook is copied from the LNL-S training notebook and uses the `LNL_Ti` model. CutMix and cooldown are temporarily disabled. Training uses only the configured affine, color jitter, ROI and Gaussian blur augmentations, with a track-safe train/validation split and standard test top-1 evaluation.

In [ ]:
# Colab/repository setup
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    colab_drive = None
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path('/content/Locality-iN-Locality')
    if not REPO_ROOT.exists():
        subprocess.run(
            ['git', 'clone', 'https://github.com/Omid-Nejati/Locality-iN-Locality.git', str(REPO_ROOT)],
            check=True,
        )
else:
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / 'LNL.py').exists():
        candidate = REPO_ROOT / 'Locality-iN-Locality-main'
        if (candidate / 'LNL.py').exists():
            REPO_ROOT = candidate
        else:
            raise FileNotFoundError('Run this notebook from the repository root on a local machine.')

os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'einops'], check=True)
print(f'Running from: {REPO_ROOT}')
print(f'Colab: {IN_COLAB}')

In [ ]:
import csv
import math
import os
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Subset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('Device:', device)

## GTSRB data preparation

In [ ]:
from urllib.request import urlretrieve
import zipfile

DATA_DIR = Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_BASE_URL = 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370'
ARCHIVES = (
    'GTSRB_Final_Training_Images.zip',
    'GTSRB_Final_Test_Images.zip',
    'GTSRB_Final_Test_GT.zip',
)
for filename in ARCHIVES:
    destination = DATA_DIR / filename
    if not destination.exists():
        print(f'Downloading {filename}...')
        urlretrieve(f'{ARCHIVE_BASE_URL}/{filename}', destination)

In [ ]:
required_paths = (
    DATA_DIR / 'GTSRB' / 'Final_Training' / 'Images',
    DATA_DIR / 'GTSRB' / 'Final_Test' / 'Images',
    DATA_DIR / 'GT-final_test.csv',
)
if not all(path.exists() for path in required_paths):
    for filename in ARCHIVES:
        with zipfile.ZipFile(DATA_DIR / filename) as archive:
            archive.extractall(DATA_DIR)
    print('GTSRB archives extracted.')
else:
    print('GTSRB dataset already extracted.')

In [ ]:
data_dir = './data/GTSRB'
images_dir = os.path.join(data_dir, 'Final_Test/Images')
test_dir = os.path.join(data_dir, 'test')
os.makedirs(test_dir, exist_ok=True)

with open('./data/GT-final_test.csv', encoding='utf-8') as f:
    image_names = f.readlines()

for text in image_names[1:]:
    class_id = int(text.split(';')[-1])
    image_name = text.split(';')[0]
    test_class_dir = os.path.join(test_dir, f'{class_id:04d}')
    os.makedirs(test_class_dir, exist_ok=True)
    shutil.copy2(os.path.join(images_dir, image_name), test_class_dir)

In [ ]:
NUM_CLASSES = 43
IMAGE_SIZE = 224
MICRO_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 0
TARGET_VALIDATION_SIZE = 4000

AUGMENTATION_CONFIG = {
    'affine': {
        'probability': 0.50,
        'degrees': 5,
        'translate': 0.02,
        'scale_min': 0.95,
        'scale_max': 1.05,
        'shear': 1,
    },
    'color_jitter': {
        'probability': 0.35,
        'brightness': 0.12,
        'contrast': 0.12,
        'saturation': 0.08,
        'hue': 0.0,
    },
    'roi': {
        'probability': 0.20,
        'border_ratio': 0.08,
    },
    'gaussian_blur': {
        'probability': 0.05,
        'kernel_size': 3,
        'sigma_min': 0.1,
        'sigma_max': 0.8,
    },
}

class ResizeWithPadding:
    def __init__(self, size=224, fill=128):
        self.size = size
        self.fill = fill

    def __call__(self, image):
        width, height = image.size
        scale = self.size / max(width, height)
        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))
        image = TF.resize(
            image,
            [new_height, new_width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )
        horizontal = self.size - new_width
        vertical = self.size - new_height
        return TF.pad(
            image,
            [horizontal // 2, vertical // 2,
             horizontal - horizontal // 2, vertical - vertical // 2],
            fill=self.fill,
        )

def crop_expanded_roi(image, box, border_ratio):
    x1, y1, x2, y2 = box
    roi_width = x2 - x1 + 1
    roi_height = y2 - y1 + 1
    pad_x = round(roi_width * border_ratio)
    pad_y = round(roi_height * border_ratio)
    left = max(0, x1 - pad_x)
    top = max(0, y1 - pad_y)
    right = min(image.width, x2 + pad_x + 1)
    bottom = min(image.height, y2 + pad_y + 1)
    if right <= left or bottom <= top:
        raise ValueError(f'Invalid ROI box: {box}')
    return image.crop((left, top, right, bottom))

def read_roi_boxes():
    boxes = {}
    for class_dir in sorted(path for path in TRAIN_ROOT.iterdir() if path.is_dir()):
        annotation = class_dir / f'GT-{class_dir.name}.csv'
        with annotation.open(newline='') as csv_file:
            for row in csv.DictReader(csv_file, delimiter=';'):
                path = (class_dir / row['Filename']).resolve()
                boxes[str(path)] = tuple(
                    int(row[key]) for key in ('Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2')
                )
    with (DATA_DIR / 'GT-final_test.csv').open(newline='') as csv_file:
        for row in csv.DictReader(csv_file, delimiter=';'):
            class_dir = TEST_ROOT / f"{int(row['ClassId']):04d}"
            path = (class_dir / row['Filename']).resolve()
            boxes[str(path)] = tuple(
                int(row[key]) for key in ('Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2')
            )
    return boxes

class ROIMixedImageFolder(torchvision.datasets.ImageFolder):
    def __init__(self, root, roi_boxes, transform, roi_probability, border_ratio):
        super().__init__(root, transform=transform)
        self.roi_boxes = roi_boxes
        self.roi_probability = roi_probability
        self.border_ratio = border_ratio

    def __getitem__(self, index):
        path, target = self.samples[index]
        image = self.loader(path)
        if random.random() < self.roi_probability:
            image = crop_expanded_roi(
                image, self.roi_boxes[str(Path(path).resolve())], self.border_ratio
            )
        if self.transform is not None:
            image = self.transform(image)
        return image, target

normalize = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
eval_transform = transforms.Compose([
    ResizeWithPadding(IMAGE_SIZE),
    transforms.ToTensor(),
    normalize,
])
affine = AUGMENTATION_CONFIG['affine']
color = AUGMENTATION_CONFIG['color_jitter']
blur = AUGMENTATION_CONFIG['gaussian_blur']
train_transform = transforms.Compose([
    ResizeWithPadding(IMAGE_SIZE),
    transforms.RandomApply([transforms.RandomAffine(
        degrees=affine['degrees'],
        translate=(affine['translate'], affine['translate']),
        scale=(affine['scale_min'], affine['scale_max']),
        shear=affine['shear'],
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=128,
    )], p=affine['probability']),
    transforms.RandomApply([transforms.ColorJitter(
        brightness=color['brightness'],
        contrast=color['contrast'],
        saturation=color['saturation'],
        hue=color['hue'],
    )], p=color['probability']),
    transforms.RandomApply([transforms.GaussianBlur(
        kernel_size=blur['kernel_size'],
        sigma=(blur['sigma_min'], blur['sigma_max'])
    )], p=blur['probability']),
    transforms.ToTensor(),
    normalize,
])

TRAIN_ROOT = DATA_DIR / 'GTSRB' / 'Final_Training' / 'Images'
TEST_ROOT = DATA_DIR / 'GTSRB' / 'test'
index_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT)
all_indices = np.arange(len(index_dataset))
all_targets = np.asarray(index_dataset.targets)

def gtsrb_track_group(sample_path):
    path = Path(sample_path)
    parts = path.stem.split('_')
    if len(parts) != 2 or not all(part.isdigit() for part in parts):
        raise ValueError(f'Unexpected GTSRB filename: {path.name}')
    return f'{path.parent.name}:{parts[0]}'

all_groups = np.asarray([
    gtsrb_track_group(path) for path, _ in index_dataset.samples
])
groups_per_class = [
    len(np.unique(all_groups[all_targets == class_id]))
    for class_id in range(len(index_dataset.classes))
]
n_splits = min(
    max(2, round(len(index_dataset) / TARGET_VALIDATION_SIZE)),
    min(groups_per_class),
)
splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
candidate_splits = list(splitter.split(all_indices, all_targets, groups=all_groups))
train_indices, validation_indices = min(
    candidate_splits, key=lambda split: abs(len(split[1]) - TARGET_VALIDATION_SIZE)
)
train_groups = set(all_groups[train_indices])
validation_groups = set(all_groups[validation_indices])
assert train_groups.isdisjoint(validation_groups), 'Track leakage detected.'
assert len(train_indices) + len(validation_indices) == len(index_dataset)

ROI_BOXES = read_roi_boxes()
training_dataset = ROIMixedImageFolder(
    TRAIN_ROOT, ROI_BOXES, train_transform,
    roi_probability=AUGMENTATION_CONFIG['roi']['probability'],
    border_ratio=AUGMENTATION_CONFIG['roi']['border_ratio'],
)
validation_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT, transform=eval_transform)
testset = torchvision.datasets.ImageFolder(TEST_ROOT, transform=eval_transform)
trainset = Subset(training_dataset, train_indices)
validationset = Subset(validation_dataset, validation_indices)
loader_options = {'num_workers': NUM_WORKERS, 'pin_memory': device.type == 'cuda'}
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(trainset, batch_size=MICRO_BATCH_SIZE, shuffle=True, generator=loader_generator, **loader_options)
validation_loader = DataLoader(validationset, batch_size=EVAL_BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(testset, batch_size=EVAL_BATCH_SIZE, shuffle=False, **loader_options)

print(f'Train: {len(trainset):,} | Validation: {len(validationset):,} | Test: {len(testset):,}')
print(f'Track-safe split: {n_splits} folds | Overlap: {len(train_groups & validation_groups)}')

## LNL-Ti model

In [ ]:
from LNL import LNL_Ti

model = LNL_Ti(pretrained=False, num_classes=NUM_CLASSES).to(device)
print(f'Model: LNL-Ti | classifier: {model.head}')

## Training augmentations (CutMix + cooldown disabled)

In [ ]:
CUTMIX_ENABLED = False
COOLDOWN_ENABLED = False
print('CutMix:', 'enabled' if CUTMIX_ENABLED else 'disabled')
print('Cooldown:', 'enabled' if COOLDOWN_ENABLED else 'disabled')

## Training: AdamW + cosine warmup scheduler

In [ ]:
class CosineWarmupScheduler(torch.optim.lr_scheduler.LambdaLR):
    def __init__(self, optimizer, warmup_steps, cosine_steps, min_lr_ratio=0.01):
        self.warmup_steps = max(1, warmup_steps)
        self.cosine_steps = max(1, cosine_steps)
        self.min_lr_ratio = min_lr_ratio
        super().__init__(optimizer, self._lr_lambda)

    def _lr_lambda(self, step):
        if step < self.warmup_steps:
            return 0.1 + 0.9 * (step + 1) / self.warmup_steps
        cosine_step = step - self.warmup_steps
        if cosine_step < self.cosine_steps:
            progress = cosine_step / self.cosine_steps
            cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
            return self.min_lr_ratio + (1.0 - self.min_lr_ratio) * cosine
        return self.min_lr_ratio

NUM_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 10
WARMUP_EPOCHS = 5
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 5e-2

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
steps_per_epoch = len(train_loader)
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
cosine_steps = max(1, (NUM_EPOCHS - WARMUP_EPOCHS) * steps_per_epoch)
scheduler = CosineWarmupScheduler(
    optimizer, warmup_steps, cosine_steps, min_lr_ratio=0.01
)
print(f'Batches/epoch: {steps_per_epoch} | Epochs: {NUM_EPOCHS} | Warmup: {WARMUP_EPOCHS}')

## Checkpoint and resume

In [ ]:
if IN_COLAB:
    colab_drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/LnL_checkpoints')
else:
    CHECKPOINT_DIR = Path('./checkpoint')

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / 'lnl_ti_gtsrb_augmentations_last.pt'
RESUME = True
start_epoch = 0
best_validation_top1 = -1.0
best_model_state = None
epochs_without_improvement = 0

if RESUME and CHECKPOINT_PATH.exists():
    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    except TypeError:  # compatibility with older PyTorch versions
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    best_validation_top1 = float(checkpoint.get('best_validation_top1', -1.0))
    best_model_state = checkpoint.get('best_model_state')
    epochs_without_improvement = int(checkpoint.get('epochs_without_improvement', 0))
    start_epoch = int(checkpoint['epoch']) + 1
    print(f'Resumed from epoch {start_epoch}/{NUM_EPOCHS}: {CHECKPOINT_PATH}')
else:
    print(f'Starting a new run; checkpoints will be saved to: {CHECKPOINT_PATH}')

In [ ]:
def cpu_state_dict(module):
    return {name: value.detach().cpu().clone() for name, value in module.state_dict().items()}

@torch.inference_mode()
def evaluate_top1(module, loader):
    module.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        logits = module(images.to(device, non_blocking=True))
        predictions = logits.argmax(dim=1).cpu()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    validation_top1 = evaluate_top1(model, validation_loader)
    current_lr = optimizer.param_groups[0]['lr']
    if validation_top1 > best_validation_top1:
        best_validation_top1 = validation_top1
        best_model_state = cpu_state_dict(model)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    torch.save({
        'epoch': epoch,
        'model_state': cpu_state_dict(model),
        'best_model_state': best_model_state,
        'best_validation_top1': best_validation_top1,
        'epochs_without_improvement': epochs_without_improvement,
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'num_classes': NUM_CLASSES,
        'augmentation_config': AUGMENTATION_CONFIG,
    }, CHECKPOINT_PATH)
    print(
        f'Epoch [{epoch + 1:03d}/{NUM_EPOCHS}] | '
        f'loss: {epoch_loss / steps_per_epoch:.4f} | '
        f'validation top-1: {validation_top1:.2f}% | '
        f'best: {best_validation_top1:.2f}% | '
        f'no improvement: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} | '
        f'lr: {current_lr:.2e}'
    )
    print(f'Checkpoint saved: {CHECKPOINT_PATH}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping after {EARLY_STOPPING_PATIENCE} epochs without validation improvement.')
        break

## Test top-1 accuracy

In [ ]:
if best_model_state is not None:
    model.load_state_dict(best_model_state)

model.eval()
correct = 0
total = 0

with torch.inference_mode():
    for images, labels in test_loader:
        logits = model(images.to(device, non_blocking=True))
        predictions = logits.argmax(dim=1).cpu()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

top1_accuracy = 100.0 * correct / total
print(f'Test top-1 accuracy: {top1_accuracy:.2f}% ({correct}/{total})')